In [ ]:
# hybrid_recommender_robust.py
# Versi robust: auto-detect column names, defensive checks, clear error messages.
import os, sys, logging, warnings
import pandas as pd
import numpy as np
from collections import defaultdict, Counter
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import log_loss, confusion_matrix, classification_report
from sklearn.metrics import pairwise_distances
from imblearn.over_sampling import RandomOverSampler, SMOTE, ADASYN
import lightgbm as lgb

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
log = logging.getLogger("hybrid_robust")

# ------------------ CONFIG ------------------
DATA_PATH = "data/dataset-telco-clean.csv"
RANDOM_STATE = 42
SMOTE_METHOD = "smote"   # "smote", "adasyn", or "none"
OUTPUT_DIR = "artifacts"
os.makedirs(OUTPUT_DIR, exist_ok=True)
PRINT_N = 50
K = 3
GRID_WEIGHTS = [(0.7,0.2,0.1),(0.6,0.25,0.15),(0.5,0.35,0.15),(0.6,0.3,0.1),(0.55,0.3,0.15)]
GRID_MIN_SCORE = [None, 0.01, 0.03, 0.05, 0.07]

# Possible names for key columns (tries in order)
POSSIBLE_TARGETS = ['target_offer','target','offer_name','offer','product','label','targetOffer','targetoffer']
POSSIBLE_ID = ['customer_id','cust_id','id','customer','customerid','customerId']

# ------------------ helpers ------------------
def find_first_existing_column(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def temperature_scaling(proba, T=1.0):
    logits = np.log(np.clip(proba, 1e-9, 1.0))
    scaled_logits = logits / T
    exp_logits = np.exp(scaled_logits)
    return exp_logits / exp_logits.sum(axis=1, keepdims=True)

def adaptive_topk_from_scores(score_row, le, k=3, min_score=None):
    idxs = np.argsort(score_row)[::-1]
    preds = []
    for idx in idxs:
        if (min_score is None) or (score_row[idx] >= min_score):
            preds.append(le.classes_[idx])
        if len(preds) == k:
            break
    if len(preds) < k:
        for idx in idxs:
            lab = le.classes_[idx]
            if lab not in preds:
                preds.append(lab)
            if len(preds) == k:
                break
    return preds

def precision_at_k(true_labels, pred_lists, k=3):
    ps = []
    for t,p in zip(true_labels,pred_lists):
        ps.append(len({t} & set(p[:k]))/k)
    return np.mean(ps)

# ------------------ LOAD ------------------
try:
    log.info("Memuat data dari: %s", DATA_PATH)
    df = pd.read_csv(DATA_PATH)
except Exception as e:
    log.exception("Gagal memuat file. Pastikan path benar. Error:")
    raise SystemExit(e)

log.info("Kolom dataset: %s", df.columns.tolist())

# auto-detect target and id
TARGET_COL = find_first_existing_column(df, POSSIBLE_TARGETS)
ID_COL = find_first_existing_column(df, POSSIBLE_ID)

if TARGET_COL is None:
    log.error("Tidak menemukan kolom target. Coba cek nama kolom. Kandidat: %s", POSSIBLE_TARGETS)
    log.error("Kolom yang tersedia: %s", df.columns.tolist())
    raise SystemExit("Target column not found.")

if ID_COL is None:
    log.warning("Tidak menemukan kolom customer_id. Akan membuat customer_id otomatis.")
    df['customer_id'] = df.index.astype(str)
    ID_COL = 'customer_id'

log.info("Menggunakan TARGET_COL=%s, ID_COL=%s", TARGET_COL, ID_COL)

# ------------------ basic cleaning ------------------
# remove rows with null target
df = df[df[TARGET_COL].notna()].copy()
df.reset_index(drop=True, inplace=True)
log.info("Setelah dropna target shape: %s", df.shape)

# find numeric feature columns to use (exclude id and target)
exclude = {TARGET_COL, ID_COL}
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
feature_candidates = [c for c in num_cols if c not in exclude]

# if not enough numeric features, try some fallback using plausible names
if len(feature_candidates) < 3:
    log.warning("Tidak cukup numeric features otomatis (found %d). Trying fallback names.", len(feature_candidates))
    fallback = ['avg_data_usage_gb','avg_call_duration','monthly_spend','topup_freq','sms_freq','pct_video_usage','complaint_count','travel_score']
    for f in fallback:
        if f in df.columns and f not in feature_candidates and f not in exclude:
            feature_candidates.append(f)
    log.info("Selected features (after fallback): %s", feature_candidates)

if len(feature_candidates) == 0:
    log.error("Tidak menemukan fitur numerik. Harus ada minimal 1 fitur numerik untuk training.")
    raise SystemExit("No numeric features found.")

# safe cast numeric and fillna
for c in feature_candidates:
    df[c] = pd.to_numeric(df[c], errors='coerce').fillna(df[c].median() if df[c].dtype != object else 0)

# Build X, y
X = df[feature_candidates].copy()
y_raw = df[TARGET_COL].astype(str).copy()
ids = df[ID_COL].astype(str).copy()

log.info("Menggunakan fitur: %s", feature_candidates)
log.info("Jumlah sampel: %d", len(df))

# label encode
le = LabelEncoder()
y = le.fit_transform(y_raw)
log.info("Label classes: %s", list(le.classes_))

# ------------------ train/test split ------------------
try:
    X_train, X_test, y_train, y_test, ids_train, ids_test = train_test_split(
        X, y, ids, test_size=0.2, random_state=RANDOM_STATE, stratify=y
    )
except Exception as e:
    log.exception("Gagal melakukan train_test_split. Error:")
    raise SystemExit(e)

log.info("Train size: %s Test size: %s", X_train.shape, X_test.shape)
log.info("Before imbalance: %s", np.bincount(y_train))

# ------------------ oversampling (minority-first) ------------------
try:
    ros = RandomOverSampler(sampling_strategy='not minority', random_state=RANDOM_STATE)
    X_tmp, y_tmp = ros.fit_resample(X_train, y_train)
    log.info("After Random Oversample: %s", np.bincount(y_tmp))

    if SMOTE_METHOD.lower() == 'adasyn':
        sm = ADASYN(random_state=RANDOM_STATE)
        X_over, y_over = sm.fit_resample(X_tmp, y_tmp)
    elif SMOTE_METHOD.lower() == 'smote':
        sm = SMOTE(k_neighbors=5, sampling_strategy='auto', random_state=RANDOM_STATE)
        X_over, y_over = sm.fit_resample(X_tmp, y_tmp)
    else:
        X_over, y_over = X_tmp, y_tmp

    log.info("After SMOTE/ADASYN: %s", np.bincount(y_over))
except Exception as e:
    log.exception("Error during oversampling:")
    raise SystemExit(e)

# ------------------ sample weights ------------------
class_w = compute_class_weight('balanced', classes=np.unique(y_over), y=y_over)
sample_weights = np.array([class_w[int(lbl)] for lbl in y_over])

# ------------------ scale ------------------
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_over)
X_test_s = scaler.transform(X_test)

# ------------------ train LightGBM ------------------
try:
    train_data = lgb.Dataset(X_train_s, label=y_over, weight=sample_weights)
    valid_data = lgb.Dataset(X_test_s, label=y_test, reference=train_data)

    params = {
        'objective': 'multiclass',
        'num_class': len(le.classes_),
        'metric': 'multi_logloss',
        'learning_rate': 0.03,
        'num_leaves': 31,
        'min_data_in_leaf': 20,
        'feature_fraction': 0.8,
        'bagging_fraction': 0.8,
        'bagging_freq': 5,
        'lambda_l1': 0.5,
        'lambda_l2': 1.0,
        'seed': RANDOM_STATE,
        'verbosity': -1,
        'n_jobs': -1
    }

    log.info("Training LightGBM (early stopping 50 rounds)...")
    bst = lgb.train(params, train_data, valid_sets=[valid_data], num_boost_round=1000,
                    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
    log.info("Model trained")
except Exception as e:
    log.exception("Training failed:")
    raise SystemExit(e)

# save artifacts minimal
joblib_path = os.path.join(OUTPUT_DIR, "hybrid_recommender_artifacts.pkl")
try:
    import joblib
    joblib.dump({'model': bst, 'label_encoder': le, 'scaler': scaler, 'feature_cols': feature_candidates}, joblib_path)
    log.info("Saved artifacts to %s", joblib_path)
except Exception:
    log.exception("Gagal menyimpan artifacts (joblib). Lanjut tanpa menyimpan.")

# ------------------ predict + temp scaling ------------------
proba_test = bst.predict(X_test_s)
# temperature search
best_T, best_nll = 1.0, np.inf
for T in [0.5,0.7,0.8,1.0,1.2]:
    try:
        p = temperature_scaling(proba_test, T=T)
        nll = log_loss(y_test, p)
        if nll < best_nll:
            best_nll, best_T = nll, T
    except Exception:
        continue
log.info("Selected temperature T = %s", best_T)
proba_test_s = temperature_scaling(proba_test, T=best_T)

# ------------------ centroids similarity ------------------
try:
    X_train_pre_sm = scaler.transform(X_train)  # scaled original X_train (before oversample)
    train_df_tmp = pd.DataFrame(X_train_pre_sm, columns=feature_candidates)
    train_df_tmp['label'] = y_train
    centroids = train_df_tmp.groupby('label')[feature_candidates].mean()
    # fill missing labels
    global_mean = train_df_tmp[feature_candidates].mean()
    for l in range(len(le.classes_)):
        if l not in centroids.index:
            centroids.loc[l] = global_mean.values
    centroids = centroids.sort_index()
    dist = pairwise_distances(X_test_s, centroids.values, metric='euclidean')
    centroid_sim = 1.0 / (1.0 + dist)
    centroid_sim = centroid_sim / (centroid_sim.max(axis=1, keepdims=True) + 1e-9)
except Exception as e:
    log.exception("Centroid similarity failed, falling back to zeros:")
    centroid_sim = np.zeros_like(proba_test_s)

# ------------------ popularity per-seg fallback (use global) ------------------
train_offer_names = le.inverse_transform(y_over)
pop_global = pd.Series(train_offer_names).value_counts(normalize=True).reindex(le.classes_, fill_value=0).values
pop_global_norm = pop_global / (pop_global.max() + 1e-9)
# simple broadcast
pop_final = np.tile(pop_global_norm, (len(X_test_s), 1))

# ------------------ grid search hybrid weights ------------------
best = {'prec': -1}
for (w_m, w_c, w_p) in GRID_WEIGHTS:
    s = w_m + w_c + w_p
    w_m_n, w_c_n, w_p_n = w_m/s, w_c/s, w_p/s
    final = (w_m_n * proba_test_s) + (w_c_n * centroid_sim) + (w_p_n * pop_final)
    for min_score in GRID_MIN_SCORE:
        try:
            preds = [adaptive_topk_from_scores(row, le, k=K, min_score=min_score) for row in final]
            prec = precision_at_k(le.inverse_transform(y_test), preds, k=K)
            if prec > best['prec']:
                best = {'prec': prec, 'weights':(w_m_n,w_c_n,w_p_n), 'min_score': min_score, 'preds': preds}
            log.info("w=%.2f,%.2f,%.2f min_score=%s => Precision@%d=%.4f", w_m_n, w_c_n, w_p_n, str(min_score), K, prec)
        except Exception:
            continue

log.info("BEST configuration: %s min_score=%s Precision@%d=%.4f", best['weights'], best['min_score'], K, best['prec'])

# ------------------ save CSV output ------------------
out_df = pd.DataFrame({
    'customer_id': ids_test.values,
    'true_offer': le.inverse_transform(y_test)
})
preds_pad = [p + [None]*(K - len(p)) if len(p) < K else p for p in best['preds']]
for i in range(K):
    out_df[f'pred_{i+1}'] = [p[i] for p in preds_pad]
out_csv = os.path.join(OUTPUT_DIR, "hybrid_recommendation_tuned_output.csv")
out_df.to_csv(out_csv, index=False)
log.info("Saved recommendations CSV to %s", out_csv)

# ------------------ PRINT first N test users ------------------
print("\n=== Sample Recommendations (first %d test users) ===" % min(PRINT_N, len(ids_test)))
for i in range(min(PRINT_N, len(ids_test))):
    uid = ids_test.iloc[i]
    true_label = le.inverse_transform([y_test[i]])[0]
    preds = best['preds'][i]
    print(f"{i+1:03d}) User {uid} | True={true_label} | Top-3={preds}")

# ------------------ global precision ------------------
print("\n=== Global Precision@%d = %.4f ===" % (K, best['prec']))

# ------------------ correct per-class precision computation ------------------
y_test_str = le.inverse_transform(y_test)
preds_all = best['preds']
hit_counts = defaultdict(int)
support = defaultdict(int)
for true_lbl, preds in zip(y_test_str, preds_all):
    support[true_lbl] += 1
    if true_lbl in preds[:K]:
        hit_counts[true_lbl] += 1

print("\n=== Precision@%d per class ===" % K)
for cls in le.classes_:
    s = support.get(cls, 0)
    hits = hit_counts.get(cls, 0)
    prec = (hits / s) if s > 0 else 0.0
    print(f"{cls}: {prec:.4f} (support={s})")

# ------------------ per-segment precision (basic) ------------------
seg_test = np.zeros(len(ids_test), dtype=int)
# if you have actual segments in your user_features, plug them here.
print("\n=== Precision@%d per segment ===" % K)
for seg in np.unique(seg_test):
    idxs = np.where(seg_test == seg)[0]
    if len(idxs) == 0:
        continue
    prec = precision_at_k([y_test_str[i] for i in idxs], [preds_all[i] for i in idxs], k=K)
    print("Segment", seg, "Precision@%d = %.4f" % (K, prec))

# ------------------ confusion matrix (top-1) & classification report ------------------
top1_preds = [p[0] for p in preds_all]
top1_true = y_test_str
print("\n=== Confusion Matrix (Top-1) ===")
cm = confusion_matrix(top1_true, top1_preds, labels=le.classes_)
print(cm)
print("Labels order:", list(le.classes_))

print("\n=== Classification Report (Top-1) ===")
print(classification_report(top1_true, top1_preds, labels=le.classes_, zero_division=0))

log.info("Selesai. Best Precision@%d = %.4f", K, best['prec'])
